# Notebook 5: Results Analysis and Thesis Figures
**MarathiMWP Thesis — Chapter 5 (Experiments & Results) + Chapter 6 (Discussion)**

This notebook:
- Compiles all experiment results into the final comparison table
- Generates all publication-quality figures for the thesis
- Performs error analysis and categorizes failure modes
- Runs statistical significance tests
- Produces case studies (good/bad examples)

**Prerequisites**: Notebooks 01–04 completed.

In [ ]:
# Install dependencies (run once)
!pip install scipy matplotlib seaborn pandas numpy sacrebleu -q

In [ ]:
import sys, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

sys.path.insert(0, str(Path('.').resolve()))
from utils.data_utils import classify_operation, extract_numbers
from utils.evaluation import (
    compute_metrics, evaluate_equation, answers_match, equations_exact_match
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

Path('figures').mkdir(exist_ok=True)
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']
print('Setup complete')

In [ ]:
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

test_data       = load_json('splits/marathi_test.json')
gold_test       = [x['Equation'] for x in test_data]

# Load all predictions
try:
    baseline_preds = load_json('splits/test_predictions_baselines.json')
    transformer_preds = load_json('splits/test_predictions_transformers.json')
    transfer_preds = load_json('splits/test_predictions_transfer.json')
    print('All prediction files loaded.')
except FileNotFoundError as e:
    print(f'Missing: {e}')
    print('Run notebooks 02-04 first to generate predictions.')

## 1. Master Results Table (Table 5.1)

In [ ]:
# Build comprehensive results table from saved predictions
def metrics_from_preds(gold, pred_key, pred_list):
    preds = [x[pred_key] for x in pred_list]
    return compute_metrics(gold, preds)

all_results = {}

# Baselines
if baseline_preds:
    all_results['Rule-Based']    = metrics_from_preds(gold_test, 'Rule',     baseline_preds)
    all_results['Template-Based']= metrics_from_preds(gold_test, 'Template', baseline_preds)
    all_results['LSTM Seq2Seq']  = metrics_from_preds(gold_test, 'LSTM',     baseline_preds)

# Transformers (monolingual)
if transformer_preds:
    all_results['mT5-small']  = metrics_from_preds(gold_test, 'mT5',       transformer_preds)
    all_results['IndicBART']  = metrics_from_preds(gold_test, 'IndicBART', transformer_preds)

# Transfer learning
if transfer_preds:
    all_results['Zero-Shot Transfer']  = metrics_from_preds(gold_test, 'ZeroShot',     transfer_preds)
    all_results['Few-Shot 50% (Hindi)']= metrics_from_preds(gold_test, 'FewShot50pct', transfer_preds)
    all_results['mT5 + Hindi (100%)']  = metrics_from_preds(gold_test, 'FewShot100pct',transfer_preds)
    all_results['Multilingual Joint']  = metrics_from_preds(gold_test, 'Multilingual', transfer_preds)

# Display
df_master = pd.DataFrame([
    {
        'Model'               : name,
        'Answer Acc (%)'      : round(m['answer_accuracy'], 1),
        'Equation Acc (%)'    : round(m['equation_accuracy'], 1),
        'Eq Equiv (%)'        : round(m['equation_equivalence'], 1),
        'BLEU'                : round(m['bleu'], 1),
    }
    for name, m in all_results.items()
])

print('\nMaster Results Table (Test Set, N=' + str(len(test_data)) + ')')
print(df_master.to_string(index=False))
df_master.to_csv('figures/table_5_master_results.csv', index=False)
print('\nSaved: figures/table_5_master_results.csv')

## 2. Model Comparison Bar Chart (Figure 5.1)

In [ ]:
model_names = list(all_results.keys())
aa_vals = [all_results[m]['answer_accuracy'] for m in model_names]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(model_names)), aa_vals, color=COLORS * 3, edgecolor='white')
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Answer Accuracy (%)', fontsize=12)
ax.set_title('Answer Accuracy Comparison Across All Models', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)

# Add value labels on bars
for bar, val in zip(bars, aa_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Add dividers between model groups
if baseline_preds and transformer_preds:
    ax.axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
    ax.text(1.0, 102, 'Baselines', ha='center', fontsize=8, color='gray')
    ax.text(3.5, 102, 'Transformers', ha='center', fontsize=8, color='gray')
    if transfer_preds:
        ax.axvline(x=4.5, color='gray', linestyle='--', alpha=0.5)
        ax.text(6.5, 102, 'Transfer Learning', ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('figures/fig_5_1_all_models_comparison.png', bbox_inches='tight')
plt.show()

## 3. Performance by Operation Type (Figure 5.2)

In [ ]:
ops = ['addition', 'subtraction', 'multiplication', 'division', 'mixed']

# Select best model from each category for comparison
selected_models = {}
if baseline_preds:
    selected_models['LSTM Seq2Seq'] = [x['LSTM'] for x in baseline_preds]
if transformer_preds:
    selected_models['mT5-small'] = [x['mT5'] for x in transformer_preds]
if transfer_preds:
    selected_models['mT5 + Hindi Transfer'] = [x['FewShot100pct'] for x in transfer_preds]

op_results = {op: {} for op in ops}
for op in ops:
    indices = [i for i, x in enumerate(test_data) if classify_operation(x['Equation']) == op]
    if not indices:
        continue
    gold_op = [gold_test[i] for i in indices]
    for model_name, preds in selected_models.items():
        pred_op = [preds[i] for i in indices]
        correct = sum(answers_match(g, p) for g, p in zip(gold_op, pred_op))
        op_results[op][model_name] = correct / len(indices) * 100

df_ops = pd.DataFrame(op_results).T
print('Answer Accuracy by Operation Type:')
print(df_ops.round(1).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ops))
n_models = len(selected_models)
width = 0.8 / n_models

for i, (model_name, color) in enumerate(zip(selected_models.keys(), COLORS)):
    vals = [op_results[op].get(model_name, 0) for op in ops]
    ax.bar(x + i * width - (n_models-1) * width / 2, vals, width,
           label=model_name, color=color)

ax.set_xticks(x)
ax.set_xticklabels([o.capitalize() for o in ops])
ax.set_ylabel('Answer Accuracy (%)')
ax.set_title('Performance by Operation Type', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('figures/fig_5_2_operation_breakdown.png', bbox_inches='tight')
plt.show()

## 4. Error Analysis

In [ ]:
def categorize_error(gold_eq, pred_eq, problem_text, n_operators):
    gold_ans = evaluate_equation(gold_eq)
    pred_ans = evaluate_equation(pred_eq)

    if answers_match(gold_eq, pred_eq):
        return 'Correct'
    if n_operators >= 2:
        return 'Multi-step Failure'
    if gold_ans is None or pred_ans is None:
        return 'Parse Error'
    gold_op = classify_operation(gold_eq)
    pred_op = classify_operation(pred_eq)
    if gold_op != pred_op:
        return 'Wrong Operation'
    nums = extract_numbers(problem_text)
    if len(nums) < 2:
        return 'Number Extraction Error'
    return 'Wrong Number Order'


error_analysis = {}
for model_name, preds in selected_models.items():
    categories = []
    for item, pred in zip(test_data, preds):
        cat = categorize_error(
            item['Equation'], pred,
            item['Problem'], item['Number of Operators'])
        categories.append(cat)
    error_analysis[model_name] = pd.Series(categories).value_counts()

df_errors = pd.DataFrame(error_analysis).fillna(0).astype(int)
print('\nError Analysis (count):')
print(df_errors.to_string())

pct = df_errors.div(len(test_data)) * 100
print('\nError Analysis (%):')
print(pct.round(1).to_string())

In [ ]:
# Error breakdown pie chart for best model
best_model = list(selected_models.keys())[-1]
error_counts = error_analysis[best_model]
error_filtered = error_counts[error_counts > 0]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    error_filtered.values, labels=error_filtered.index,
    autopct='%1.1f%%', colors=COLORS[:len(error_filtered)],
    startangle=140, textprops={'fontsize': 10})
ax.set_title(f'Error Analysis — {best_model}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_5_4_error_analysis.png', bbox_inches='tight')
plt.show()

## 5. Case Studies (Good + Bad Examples)

In [ ]:
if transfer_preds:
    best_preds = [x['FewShot100pct'] for x in transfer_preds]
elif transformer_preds:
    best_preds = [x['mT5'] for x in transformer_preds]
else:
    best_preds = [x['LSTM'] for x in baseline_preds]

# Good examples (model got it right)
good_examples = [
    (item, pred) for item, pred in zip(test_data, best_preds)
    if answers_match(item['Equation'], pred)
][:5]

# Bad examples (model got it wrong)
bad_examples = [
    (item, pred) for item, pred in zip(test_data, best_preds)
    if not answers_match(item['Equation'], pred)
][:5]

print('=== GOOD PREDICTIONS ===')
for item, pred in good_examples:
    gold_ans = evaluate_equation(item['Equation'])
    pred_ans = evaluate_equation(pred)
    print(f'\nProblem : {item["Problem"]}')
    print(f'Gold    : {item["Equation"]}  (ans={gold_ans})')
    print(f'Pred    : {pred}  (ans={pred_ans})  ✓')

print('\n=== BAD PREDICTIONS ===')
for item, pred in bad_examples:
    gold_ans = evaluate_equation(item['Equation'])
    pred_ans = evaluate_equation(pred)
    print(f'\nProblem : {item["Problem"]}')
    print(f'Gold    : {item["Equation"]}  (ans={gold_ans})')
    print(f'Pred    : {pred}  (ans={pred_ans})  ✗')

## 6. Statistical Significance Tests

In [ ]:
# McNemar's test: compare two model predictions on same test set
def mcnemar_test(gold, preds_a, preds_b):
    """Return p-value for McNemar's test comparing two sets of predictions."""
    correct_a = np.array([answers_match(g, p) for g, p in zip(gold, preds_a)])
    correct_b = np.array([answers_match(g, p) for g, p in zip(gold, preds_b)])

    # Contingency table
    b = ((correct_a == 1) & (correct_b == 0)).sum()  # A correct, B wrong
    c = ((correct_a == 0) & (correct_b == 1)).sum()  # A wrong, B correct

    # McNemar's statistic
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1)**2 / (b + c)
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
    return p_val


if transfer_preds and transformer_preds:
    mt5_preds_list     = [x['mT5']          for x in transformer_preds]
    transfer_preds_list= [x['FewShot100pct'] for x in transfer_preds]
    joint_preds_list   = [x['Multilingual']  for x in transfer_preds]

    p_mt5_vs_transfer = mcnemar_test(gold_test, mt5_preds_list, transfer_preds_list)
    p_mt5_vs_joint    = mcnemar_test(gold_test, mt5_preds_list, joint_preds_list)

    print('McNemar\'s Test (p-values):')
    print(f'  mT5 vs. mT5+Hindi Transfer : p = {p_mt5_vs_transfer:.4f}  {"*" if p_mt5_vs_transfer < 0.05 else "n.s."}')
    print(f'  mT5 vs. Multilingual Joint : p = {p_mt5_vs_joint:.4f}  {"*" if p_mt5_vs_joint < 0.05 else "n.s."}')
    print('  (* = significant at p < 0.05)')

## 7. All Figures Summary

In [ ]:
figures = list(Path('figures').glob('*.png')) + list(Path('figures').glob('*.csv'))
print('Generated files for thesis:')
for f in sorted(figures):
    print(f'  {f.name}')

print('\n\nThesis Chapter Mapping:')
print('  Chapter 3 (Dataset)    : table_3_2, fig_3_1, fig_3_2, fig_3_3, fig_3_4')
print('  Chapter 5 (Results)    : table_5_master, fig_5_1, fig_5_2, fig_5_3, fig_5_4')
print('  Chapter 5 (Baselines)  : table_5_1, fig_5_lstm_training_curve')
print('  Chapter 5 (Transfer)   : table_5_3')